# Session 10: Gradient descent for a line fit

**KLAi | Mathematics for AI and Machine Learning**

In this lab you fit a line to eight boda boda trips by gradient descent, with a loop you write yourself, and you watch what four different learning rates do to the loss. The same numbers are in the spreadsheet lab. The trip data are illustrative (made-up numbers to show the arithmetic).

The five steps match the lab map on the slides. Each step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE one thing**. A cell that says `None  # your code here` is yours to fill. The check cells say OK or NOT YET.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Look at the data and centre the distance

Each row is one trip: the distance in km and the fare in thousand UGX. Gradient descent works best when the input is centred, so we subtract the mean distance from every distance. The centred distance is called `xc`, and it uses the mean you compute.

**PREDICT first.** The distances run from 1 to 8 km. What is their mean?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({"distance_km": [1, 2, 3, 4, 5, 6, 7, 8],
                   "fare": [3.2, 4.1, 5.4, 5.9, 7.6, 8.0, 9.9, 10.3]})   # fare in thousand UGX
print(df)

In [ ]:
# Hint: use the mean of the distance column
mean_distance = None  # your code here

In [ ]:
x = df["distance_km"].to_numpy(dtype=float)
y = df["fare"].to_numpy(dtype=float)
n = len(x)
check("mean distance is 4.5", mean_distance, 4.5)
if mean_distance is None:
    print("Fill in mean_distance in the cell above first: the centred distance is built from it.")
    xc = x - x.mean()                      # a stand-in so that later cells can still run
else:
    xc = x - mean_distance                 # the centred distance, built from your mean
    print("centred distances:", xc)

**READ the output.** Why do the centred distances run from -3.5 to +3.5? What does a centred distance of 0 mean for a trip?

> My answer: 

---
## Step 2: The loss and the gradient at the start

The model is y-hat = b + w * xc. We start with w = 0 and b = 0. The loss is the mean squared error. The gradient has two entries: dL/dw = (2/n) * sum(err * xc) and dL/db = (2/n) * sum(err), where err = y-hat - y.

**PREDICT first.** With w = 0 and b = 0 every prediction is 0. So what is the loss? (Hint: the mean of y squared.)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: err is y-hat minus y. Use np.mean for the averages.
w, b = 0.0, 0.0
err = w * xc + b - y
loss0 = None  # your code here: the mean of err squared
dw0 = None    # your code here: 2 times the mean of err * xc
db0 = None    # your code here: 2 times the mean of err

In [ ]:
check("loss at w = 0, b = 0", loss0, 52.1100, 0.01)
check("dL/dw at the start", dw0, -11.0250, 0.01)
check("dL/db at the start", db0, -13.6000, 0.01)
if loss0 is not None:
    print("loss at the start:", loss0, " slopes:", dw0, db0)

**READ the output.** Both slopes are negative. Which way should w and b move to lower the loss?

> My answer: 

---
## Step 3: One step by hand

The update rule is new = old - eta * slope. Use eta = 0.1.

**PREDICT first.** Will the new w and b be bigger or smaller than 0? Roughly how big?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: apply the update rule from the slides to w and to b
eta = 0.1
w1 = None  # your code here
b1 = None  # your code here

In [ ]:
check("w after one step", w1, 1.1025, 0.001)
check("b after one step", b1, 1.3600, 0.001)
if w1 is not None:
    print("after one step:", w1, b1)

**CHANGE one thing.** Change eta to 0.2 and run Step 3 again. How do w1 and b1 change? Put eta back to 0.1 before you go on.

> What I saw: 

---
## Step 4: Write the loop, run 30 steps, compare with least squares

The frame of the loop is given: it starts at w = 0 and b = 0, computes the error of every row, and records the loss after every step. You write the four lines in the middle: the two slopes and the two updates. Then compare the result with the least squares line of Session 4 for these eight trips: `np.polyfit(xc, y, 1)` returns the slope and then the intercept.

**PREDICT first.** After 30 steps with eta = 0.1, will gradient descent reach the least squares line, or stop short of it?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def gradient_descent(eta, steps=30):
    """Run gradient descent on the centred line fit. Returns the final w, b and the loss at every step."""
    w, b = 0.0, 0.0
    losses = [np.mean((w * xc + b - y) ** 2)]
    for step in range(steps):
        err = w * xc + b - y                  # the error y-hat - y of every row
        dw = None      # your code here: the slope for w, the same formula as dw0 in step 2
        db = None      # your code here: the slope for b, the same formula as db0 in step 2
        w_new = None   # your code here: the update rule of step 3, applied to w
        b_new = None   # your code here: the update rule of step 3, applied to b
        if w_new is None or b_new is None:
            return None, None, None             # the loop body is not written yet
        w, b = w_new, b_new
        losses.append(np.mean((w * xc + b - y) ** 2))
    return w, b, losses

w_fit, b_fit, losses = gradient_descent(0.1)
if losses is None:
    print("Write the four lines inside the loop, then run this cell again.")
else:
    print("w =", round(w_fit, 4), " b =", round(b_fit, 4))
    print("loss at steps 0, 1, 5, 30:", [round(losses[k], 4) for k in (0, 1, 5, 30)])

In [ ]:
# Hint: np.polyfit fits a polynomial; a line is a polynomial of degree 1. It returns the slope first
w_ls, b_ls = None, None  # your code here

In [ ]:
check("loss after 30 steps, eta = 0.1", None if losses is None else losses[-1], 0.0819, 0.001)
check("least squares slope", w_ls, 1.050, 0.01)
check("least squares intercept", b_ls, 6.800, 0.01)
check("gradient descent is within 0.01 of the least squares slope", w_fit, 1.050, 0.01)
if w_ls is not None and w_fit is not None:
    print("least squares: w =", round(w_ls, 4), " b =", round(b_ls, 4), "| gradient descent: w =", round(w_fit, 4), " b =", round(b_fit, 4))

**READ the output.** Is the loss going down at every step? The intercept b is the fare for a trip of average length (4.5 km). Is it close to the mean of the fares?

> My answer: 

---
## Step 5: Four learning rates, four loss curves

Run your loop with eta = 0.01, 0.05, 0.1 and 0.2 and plot the four loss curves. The vertical axis uses a log scale so that all four fit.

**PREDICT first.** Which curve falls fastest? Which curve will go up instead of down? Write both guesses first.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
etas = [0.01, 0.05, 0.1, 0.2]
curves = {}
for eta_k in etas:
    wk, bk, lk = gradient_descent(eta_k)
    curves[eta_k] = lk

if curves[0.1] is None:
    print("Finish the loop in step 4 first.")
else:
    for eta_k in etas:
        plt.plot(curves[eta_k], label="eta = " + str(eta_k))
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("loss (log scale)")
    plt.legend()
    plt.title("Loss curves for four learning rates")
    plt.show()

In [ ]:
# Hint: each list holds the loss at every step; you want the last entry of each, and for "grew" compare the last entry of the eta = 0.2 list with its first
loss_001 = None  # your code here: loss after 30 steps for eta = 0.01
loss_005 = None  # your code here: eta = 0.05
loss_02 = None   # your code here: eta = 0.2
grew = None      # your code here: True if the eta = 0.2 loss ended higher than it started

In [ ]:
check("loss after 30 steps, eta = 0.01", loss_001, 13.8482, 0.01)
check("loss after 30 steps, eta = 0.05", loss_005, 0.1650, 0.01)
check("loss after 30 steps, eta = 0.2", loss_02, 1762.46, 0.1)
check("eta = 0.2 loss grew", grew, 1, 0.01)
if loss_02 is not None:
    print(loss_001, loss_005, loss_02, grew)

**READ the output.** Which eta makes the loss grow? Why does a step that is too large make the loss bigger? (Think of the bowl picture and overshooting.)

> My answer: 

**CHANGE one thing.** Type a learning rate of your own into the cell below, for example 0.19, which sits close to the limit between falling and growing. Run it and read the losses at steps 0, 10 and 30.

> What I saw: 

In [ ]:
my_eta = None   # type a learning rate here, for example 0.19
if my_eta is None or curves[0.1] is None:
    print("Type a number for my_eta (and finish step 4) first.")
else:
    _, _, lk = gradient_descent(my_eta)
    print([round(lk[k], 4) for k in (0, 10, 30)])

**CHANGE one thing.** Do not centre the distance: in step 1 replace `x - mean_distance` by `x`, run steps 1 to 5 again and see which etas still work. Write what you saw, then undo the change.

> What I saw: 

---
## Finish

Run the cell below. It should say **All checks pass**. Then restart the kernel, run all cells from the top, and save the notebook.

In [ ]:
show_checks(14)